In [3]:
"""
End-user LPG price allocation per pixel (Nigeria, EPSG:3857)

Raster-to-raster implementation:
- input: multilayer pixel raster from 4.2
- output: copied multilayer raster + 4 new price bands
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

DATA_DIR = Path("dataset_big")

# Input from 4.2 (multilayer raster with 6 bands)
SOURCE_PIXEL_RASTER = DATA_DIR / "huff_preferred_distributor_per_pixel.tif"

# Cost source from 4.4 / 4.5
COST_SOURCE_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
RESELLER_ID_COL = "id_res&fil"
RESELLER_COST_COL = "cost_res_kg_out"
FILLING_LAYER = "filling"
FILLING_COST_COL = "cost_fil_kg_out"

# Input from 4.3 and base demography
LPG_USE_SHARE_RASTER = DATA_DIR / "lpg_use_share.tif"
POPULATION_RASTER = DATA_DIR / "Population.tif"

# Optional spatial VOT input from 4.1
INCOME_RASTER = DATA_DIR / "income_nigeria.tif"
USE_SPATIAL_VOT = True

# Output raster
OUTPUT_PIXEL_RASTER = DATA_DIR / "real_end_user_price.tif"

# New output band names
OUT_COST_REF_WALK = "res_cost_kg_out_walk_ref"
OUT_COST_REF_CAR = "res_cost_kg_out_car_ref"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"

# Sentinel and guards
SENTINEL = 9999.0
NODATA_FLOAT = -9999.0

# Walking collection model
WALK_TIME_IS_ONE_WAY = True
MAX_ACCEPTABLE_WALK_HOURS = 7.0
FIXED_TIME_AT_RETAILER_HOURS = 0.25
CYLINDER_WEIGHT_KG = 12.5
WALK_ELIGIBILITY_THRESHOLD_MIN = 30.0

# Driving collection model
VAN_VARIABLE_COST_PER_KM = 0.437  # add source
VAN_SPEED_KPH = 15.0
MAX_ACCEPTABLE_DRIVE_HOURS = 4.0
CAR_POOL_SIZE_PERSONS = 4.0
CAR_ELIGIBILITY_THRESHOLD_MIN = 45.0

# LPG demand assumptions
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY_LPG = 0.5
ENERGY_CONTENT_LPG_MJ_PER_KG = 45.5

# VOT
MINIMUM_WAGE_USD_PER_MONTH = 26.0  # verify this number
WORK_HOURS_PER_MONTH = 30 * 8
WAGE_RANGE = (0.2, 0.5)
WAGE_FRACTION_FALLBACK = 0.3
DEFAULT_VOT_USD_PER_HOUR = WAGE_FRACTION_FALLBACK * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
VOT_TODO_NOTE = "TODO: review MINIMUM_WAGE_USD_PER_MONTH and WAGE_RANGE for local context"


def _annual_lpg_demand_kg(num_clients: np.ndarray) -> np.ndarray:
    annual_energy_mj = num_clients * MEALS_PER_DAY * 365.0 * ENERGY_PER_MEAL_MJ
    annual_mass_kg = annual_energy_mj / (EFFICIENCY_LPG * ENERGY_CONTENT_LPG_MJ_PER_KG)
    return annual_mass_kg


def _read_single_band(path: Path) -> tuple[np.ndarray, dict]:
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        profile = src.profile.copy()
        nodata = src.nodata
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    return arr, profile


def _align_to_reference(path: Path, ref_profile: dict, resampling: Resampling) -> np.ndarray:
    with rasterio.open(path) as src:
        dst = np.full((ref_profile["height"], ref_profile["width"]), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def _compute_spatial_vot(income_array: np.ndarray) -> tuple[np.ndarray, str]:
    valid = np.isfinite(income_array)
    vot = np.full(income_array.shape, DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    if not np.any(valid):
        return vot, "fallback_fixed_no_valid_income"

    min_value = float(np.nanmin(income_array[valid]))
    max_value = float(np.nanmax(income_array[valid]))
    if not np.isfinite(min_value) or not np.isfinite(max_value) or max_value <= min_value:
        return vot, "fallback_fixed_flat_income"

    wage_min, wage_max = WAGE_RANGE
    norm = (income_array - min_value) / (max_value - min_value)
    norm = np.clip(norm, 0.0, 1.0)
    wage_factor = wage_min + norm * float(wage_max - wage_min)

    vot_valid = wage_factor * (MINIMUM_WAGE_USD_PER_MONTH / WORK_HOURS_PER_MONTH)
    vot[valid] = vot_valid[valid].astype(np.float32)
    return vot, "spatial_income"


def _map_effective_cost(ids_int: np.ndarray, map_resell: dict[int, float], map_filling: dict[int, float]) -> np.ndarray:
    out = np.full(ids_int.shape, np.nan, dtype=np.float64)
    valid = ids_int > 0
    if not np.any(valid):
        return out

    uniq, inv = np.unique(ids_int[valid], return_inverse=True)
    mapped = np.fromiter((map_resell.get(int(rid), np.nan) for rid in uniq), dtype=np.float64, count=uniq.size)
    miss = ~np.isfinite(mapped)
    if np.any(miss):
        miss_ids = uniq[miss]
        mapped[miss] = np.fromiter((map_filling.get(int(rid), np.nan) for rid in miss_ids), dtype=np.float64, count=miss_ids.size)

    out_valid = mapped[inv]
    out[valid] = out_valid
    return out


def _write_multiband(path: Path, base_profile: dict, bands: list[np.ndarray], names: list[str]) -> None:
    if len(bands) != len(names):
        raise ValueError("Bands and names must have same length.")
    profile = base_profile.copy()
    profile.update(dtype="float32", count=len(bands), nodata=NODATA_FLOAT, compress="lzw")

    with rasterio.open(path, "w", **profile) as dst:
        for i, (band, name) in enumerate(zip(bands, names), start=1):
            out = np.where(np.isfinite(band), band, NODATA_FLOAT).astype(np.float32)
            dst.write(out, i)
            dst.set_band_description(i, name)


print("[1/8] Reading source multilayer pixel raster...")
if not SOURCE_PIXEL_RASTER.exists():
    raise FileNotFoundError(f"Missing input raster from 4.2: {SOURCE_PIXEL_RASTER}")

with rasterio.open(SOURCE_PIXEL_RASTER) as src:
    if src.count < 6:
        raise RuntimeError(f"Expected at least 6 bands in {SOURCE_PIXEL_RASTER}, found {src.count}")
    base_stack = src.read(indexes=[1, 2, 3, 4, 5, 6]).astype(np.float32, copy=False)
    base_profile = src.profile.copy()
    src_nodata = src.nodata

if src_nodata is not None:
    base_stack = np.where(base_stack == src_nodata, np.nan, base_stack).astype(np.float32)

car_share = base_stack[0]
walk_share = base_stack[1]
walk_id = base_stack[2]
walk_time_min = base_stack[3]
car_id = base_stack[4]
car_time_min = base_stack[5]

height, width = car_share.shape
print(f"Source grid: {width} x {height}")

print("[2/8] Checking external inputs...")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Missing cost source GPKG: {COST_SOURCE_GPKG}")
if not LPG_USE_SHARE_RASTER.exists():
    raise FileNotFoundError(f"Missing 4.3 output raster: {LPG_USE_SHARE_RASTER}")
if not POPULATION_RASTER.exists():
    raise FileNotFoundError(f"Missing population raster: {POPULATION_RASTER}")

print("[3/8] Loading cost layers and creating fallback cost maps...")
resell = gpd.read_file(COST_SOURCE_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(COST_SOURCE_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")
for c in [RESELLER_ID_COL, RESELLER_COST_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}'")
for c in [RESELLER_ID_COL, FILLING_COST_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing '{c}' in layer '{FILLING_LAYER}'")

resell_id = pd.to_numeric(resell[RESELLER_ID_COL], errors="coerce")
resell_cost = pd.to_numeric(resell[RESELLER_COST_COL], errors="coerce")
fill_id = pd.to_numeric(filling[RESELLER_ID_COL], errors="coerce")
fill_cost = pd.to_numeric(filling[FILLING_COST_COL], errors="coerce")

mask_resell = resell_id.notna()
mask_fill = fill_id.notna()
map_resell_cost = dict(zip(resell_id[mask_resell].astype(np.int64), resell_cost[mask_resell].astype(float)))
map_fill_cost = dict(zip(fill_id[mask_fill].astype(np.int64), fill_cost[mask_fill].astype(float)))

print("[4/8] Loading population/LPG/VOT rasters...")
lpg_use_share, _ = _read_single_band(LPG_USE_SHARE_RASTER)
population, pop_profile = _read_single_band(POPULATION_RASTER)
if lpg_use_share.shape != (height, width) or population.shape != (height, width):
    raise RuntimeError("Shape mismatch among source pixel raster, LPG use raster, and population raster.")

if USE_SPATIAL_VOT and INCOME_RASTER.exists():
    income_aligned = _align_to_reference(INCOME_RASTER, pop_profile, Resampling.bilinear)
    vot_raster, vot_mode = _compute_spatial_vot(income_aligned)
elif USE_SPATIAL_VOT and (not INCOME_RASTER.exists()):
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fallback_fixed_income_missing"
else:
    vot_raster = np.full((height, width), DEFAULT_VOT_USD_PER_HOUR, dtype=np.float32)
    vot_mode = "fixed_user_parameter"

print("[5/8] Computing reference costs and channel demands...")
n = height * width

walk_id_flat = walk_id.reshape(-1).astype(np.float64)
car_id_flat = car_id.reshape(-1).astype(np.float64)
walk_time_flat = walk_time_min.reshape(-1).astype(np.float64)
car_time_flat = car_time_min.reshape(-1).astype(np.float64)
share_walk_flat = walk_share.reshape(-1).astype(np.float64)
share_car_flat = car_share.reshape(-1).astype(np.float64)
pop_flat = population.reshape(-1).astype(np.float64)
lpg_flat = lpg_use_share.reshape(-1).astype(np.float64)
vot_flat = vot_raster.reshape(-1).astype(np.float64)

walk_id_int = np.where(np.isfinite(walk_id_flat) & (walk_id_flat > 0), walk_id_flat.astype(np.int64), -1)
car_id_int = np.where(np.isfinite(car_id_flat) & (car_id_flat > 0), car_id_flat.astype(np.int64), -1)

ref_cost_walk_arr = _map_effective_cost(walk_id_int, map_resell_cost, map_fill_cost)
ref_cost_car_arr = _map_effective_cost(car_id_int, map_resell_cost, map_fill_cost)

share_walk_arr = np.clip(share_walk_flat, 0.0, 1.0)
share_car_arr = np.clip(share_car_flat, 0.0, 1.0)
share_sum = share_walk_arr + share_car_arr
share_valid = share_sum > 0
share_walk_norm = np.zeros(n, dtype=np.float64)
share_car_norm = np.zeros(n, dtype=np.float64)
share_walk_norm[share_valid] = share_walk_arr[share_valid] / share_sum[share_valid]
share_car_norm[share_valid] = share_car_arr[share_valid] / share_sum[share_valid]

walk_round_trip_min = np.where(WALK_TIME_IS_ONE_WAY, 2.0 * walk_time_flat, walk_time_flat)
walk_time_hours = walk_round_trip_min / 60.0

walk_eligible = np.isfinite(walk_time_flat) & (walk_time_flat <= WALK_ELIGIBILITY_THRESHOLD_MIN)
car_eligible = np.isfinite(car_time_flat) & (car_time_flat <= CAR_ELIGIBILITY_THRESHOLD_MIN)
eligible_weight = share_walk_norm * walk_eligible.astype(np.float64) + share_car_norm * car_eligible.astype(np.float64)



total_clients = np.where(
    np.isfinite(pop_flat) & np.isfinite(lpg_flat) & (pop_flat > 0) & (lpg_flat >= 0),
    pop_flat * lpg_flat,
    np.nan,
).astype(np.float64)






# Walker model
walk_frac = np.zeros(n, dtype=np.float64)
ew_valid = eligible_weight > 0
walk_frac[ew_valid] = share_walk_norm[ew_valid] * walk_eligible[ew_valid].astype(np.float64) / eligible_weight[ew_valid]
walk_frac = np.clip(walk_frac, 0.0, 1.0)

walk_clients = np.where(np.isfinite(total_clients), total_clients * walk_frac, np.nan).astype(np.float64)
walk_clients = np.where(np.isfinite(walk_clients), np.maximum(walk_clients, 0.0), np.nan)

annual_demand_walk_kg = _annual_lpg_demand_kg(walk_clients)
trips_per_year_walk = np.where(
    np.isfinite(annual_demand_walk_kg) & (annual_demand_walk_kg > 0),
    np.ceil(annual_demand_walk_kg / CYLINDER_WEIGHT_KG),
    np.nan,
).astype(np.float64)

infeasible_walk = np.isfinite(walk_time_hours) & (walk_time_hours > MAX_ACCEPTABLE_WALK_HOURS)
time_per_trip_hours_walk = walk_time_hours + FIXED_TIME_AT_RETAILER_HOURS
total_annual_hours_walk = trips_per_year_walk * time_per_trip_hours_walk
annual_walk_cost_usd = total_annual_hours_walk * vot_flat

collection_cost_per_kg_walk = np.divide(
    annual_walk_cost_usd,
    annual_demand_walk_kg,
    out=np.full(n, np.nan, dtype=np.float64),
    where=np.isfinite(annual_demand_walk_kg) & (annual_demand_walk_kg > 0),
)

valid_walk_mask = (
    np.isfinite(ref_cost_walk_arr)
    & (walk_id_int > 0)
    & np.isfinite(walk_time_hours)
    & (walk_time_hours >= 0)
    & (~infeasible_walk)
    & np.isfinite(collection_cost_per_kg_walk)
    & (collection_cost_per_kg_walk >= 0)
    & np.isfinite(annual_demand_walk_kg)
    & (annual_demand_walk_kg > 0)
    & np.isfinite(vot_flat)
)

cost_walk_final = np.full(n, float(SENTINEL), dtype=np.float64)
cost_walk_final[valid_walk_mask] = ref_cost_walk_arr[valid_walk_mask] + collection_cost_per_kg_walk[valid_walk_mask]

# Driver model (pooled)
car_frac = np.zeros(n, dtype=np.float64)
car_frac[ew_valid] = share_car_norm[ew_valid] * car_eligible[ew_valid].astype(np.float64) / eligible_weight[ew_valid]
car_frac = np.clip(car_frac, 0.0, 1.0)

car_clients = np.where(np.isfinite(total_clients), total_clients * car_frac, np.nan).astype(np.float64)
car_clients = np.where(np.isfinite(car_clients), np.maximum(car_clients, 0.0), np.nan)

annual_demand_car_total_kg = _annual_lpg_demand_kg(car_clients)
annual_demand_per_person_kg = float(_annual_lpg_demand_kg(np.array([1.0], dtype=np.float64))[0])
annual_demand_per_group_kg = annual_demand_per_person_kg * CAR_POOL_SIZE_PERSONS
trips_per_group_per_year = float(np.ceil(annual_demand_per_group_kg / CYLINDER_WEIGHT_KG))

round_trip_drive_hours = (car_time_flat * 2.0) / 60.0
total_trip_time_hours_car = round_trip_drive_hours + FIXED_TIME_AT_RETAILER_HOURS
round_trip_distance_km_car = round_trip_drive_hours * VAN_SPEED_KPH

vehicle_operating_cost_trip_usd = round_trip_distance_km_car * VAN_VARIABLE_COST_PER_KM
driver_time_cost_trip_usd = total_trip_time_hours_car * vot_flat
trip_cost_per_vehicle_trip_usd = vehicle_operating_cost_trip_usd + driver_time_cost_trip_usd

total_groups_car = car_clients / CAR_POOL_SIZE_PERSONS
total_trips_car = total_groups_car * trips_per_group_per_year
annual_transport_cost_car_total_usd = total_trips_car * trip_cost_per_vehicle_trip_usd

collection_cost_per_kg_car = np.divide(
    annual_transport_cost_car_total_usd,
    annual_demand_car_total_kg,
    out=np.full(n, np.nan, dtype=np.float64),
    where=np.isfinite(annual_demand_car_total_kg) & (annual_demand_car_total_kg > 0),
)

infeasible_drive = np.isfinite(total_trip_time_hours_car) & (total_trip_time_hours_car > MAX_ACCEPTABLE_DRIVE_HOURS)
valid_driver_mask = (
    np.isfinite(ref_cost_car_arr)
    & (car_id_int > 0)
    & np.isfinite(car_time_flat)
    & (car_time_flat >= 0)
    & np.isfinite(total_trip_time_hours_car)
    & (~infeasible_drive)
    & np.isfinite(collection_cost_per_kg_car)
    & (collection_cost_per_kg_car >= 0)
    & np.isfinite(annual_demand_car_total_kg)
    & (annual_demand_car_total_kg > 0)
    & np.isfinite(vot_flat)
)

cost_car_final = np.full(n, float(SENTINEL), dtype=np.float64)
cost_car_final[valid_driver_mask] = ref_cost_car_arr[valid_driver_mask] + collection_cost_per_kg_car[valid_driver_mask]

print("[6/8] Writing output multilayer raster...")

best_id_walk_band = np.where(np.isfinite(walk_id) & (walk_id >= 0), walk_id, np.nan)
best_id_car_band = np.where(np.isfinite(car_id) & (car_id >= 0), car_id, np.nan)

out_ref_walk = ref_cost_walk_arr.reshape(height, width)
out_ref_car = ref_cost_car_arr.reshape(height, width)
out_cost_walk = cost_walk_final.reshape(height, width)
out_cost_car = cost_car_final.reshape(height, width)

out_bands = [
    car_share.astype(np.float32),
    walk_share.astype(np.float32),
    best_id_walk_band.astype(np.float32),
    walk_time_min.astype(np.float32),
    best_id_car_band.astype(np.float32),
    car_time_min.astype(np.float32),
    out_ref_walk.astype(np.float32),
    out_ref_car.astype(np.float32),
    out_cost_walk.astype(np.float32),
    out_cost_car.astype(np.float32),
]

out_names = [
    "car_share",
    "walk_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_reseller_id_car",
    "best_time_car_min",
    OUT_COST_REF_WALK,
    OUT_COST_REF_CAR,
    OUT_COST_WALK,
    OUT_COST_CAR,
]

_write_multiband(OUTPUT_PIXEL_RASTER, base_profile, out_bands, out_names)

print("[7/8] Diagnostics...")
n_total = n
n_valid_walk = int(valid_walk_mask.sum())
n_sentinel_walk = int(np.isclose(cost_walk_final, SENTINEL).sum())
n_infeasible_walk = int(infeasible_walk.sum())
n_positive_walk_clients = int(np.sum(np.isfinite(walk_clients) & (walk_clients > 0)))

n_valid_driver = int(valid_driver_mask.sum())
n_sentinel_driver = int(np.isclose(cost_car_final, SENTINEL).sum())
n_infeasible_driver = int(infeasible_drive.sum())
n_positive_car_clients = int(np.sum(np.isfinite(car_clients) & (car_clients > 0)))

print("=== WALKER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid walker cost                     : {n_valid_walk:,}/{n_total:,}")
print(f"Rows sentinel ({SENTINEL:.0f})                    : {n_sentinel_walk:,}/{n_total:,}")
print(f"Rows infeasible walk (> {MAX_ACCEPTABLE_WALK_HOURS:.1f} h RT): {n_infeasible_walk:,}")
print(f"Rows with positive reconstructed walk clients: {n_positive_walk_clients:,}")

print("\n=== DRIVER COST MODEL SUMMARY ===")
print(f"Rows total                                 : {n_total:,}")
print(f"Rows valid driver cost                     : {n_valid_driver:,}/{n_total:,}")
print(f"Rows sentinel ({SENTINEL:.0f})                    : {n_sentinel_driver:,}/{n_total:,}")
print(f"Rows infeasible drive (> {MAX_ACCEPTABLE_DRIVE_HOURS:.1f} h total): {n_infeasible_driver:,}")
print(f"Rows with positive reconstructed car clients : {n_positive_car_clients:,}")
print(f"Car pooling size (persons/trip)             : {CAR_POOL_SIZE_PERSONS:.1f}")
print(f"Trips per pooled group per year (driver)    : {trips_per_group_per_year:.0f}")

print(f"\nVOT mode                                   : {vot_mode}")
if np.any(np.isfinite(vot_flat)):
    v = vot_flat[np.isfinite(vot_flat)]
    print(f"VOT USD/h min/median/mean/p95/max         : {np.nanmin(v):.6f} / {np.nanmedian(v):.6f} / {np.nanmean(v):.6f} / {np.nanpercentile(v, 95):.6f} / {np.nanmax(v):.6f}")
print(VOT_TODO_NOTE)

print("[8/8] Done.")
print(f"Output file: {OUTPUT_PIXEL_RASTER}")
print(f"Bands written: {', '.join(out_names)}")
print("Walker formula:")
print("  cost_kg_walker = reference_point_cost + collection_cost_per_kg_walk")
print("Driver formula:")
print("  cost_kg_driver = reference_point_cost + collection_cost_per_kg_car")
print(f"Primary reseller source column: {RESELLER_COST_COL}")
print(f"Fallback filling source column: {FILLING_COST_COL}")
print(f"Sentinel for invalid assignments: {SENTINEL}")

[1/8] Reading source multilayer pixel raster...
Source grid: 1337 x 1085
[2/8] Checking external inputs...
[3/8] Loading cost layers and creating fallback cost maps...
[4/8] Loading population/LPG/VOT rasters...
[5/8] Computing reference costs and channel demands...


C:\Users\matti\AppData\Local\Temp\ipykernel_3820\638471447.py:250: RuntimeWarning: invalid value encountered in cast
  walk_id_int = np.where(np.isfinite(walk_id_flat) & (walk_id_flat > 0), walk_id_flat.astype(np.int64), -1)
C:\Users\matti\AppData\Local\Temp\ipykernel_3820\638471447.py:251: RuntimeWarning: invalid value encountered in cast
  car_id_int = np.where(np.isfinite(car_id_flat) & (car_id_flat > 0), car_id_flat.astype(np.int64), -1)


[6/8] Writing output multilayer raster...
[7/8] Diagnostics...
=== WALKER COST MODEL SUMMARY ===
Rows total                                 : 1,450,645
Rows valid walker cost                     : 12,847/1,450,645
Rows sentinel (9999)                    : 1,437,798/1,450,645
Rows infeasible walk (> 7.0 h RT): 432,907
Rows with positive reconstructed walk clients: 12,847

=== DRIVER COST MODEL SUMMARY ===
Rows total                                 : 1,450,645
Rows valid driver cost                     : 15,436/1,450,645
Rows sentinel (9999)                    : 1,435,209/1,450,645
Rows infeasible drive (> 4.0 h total): 499,623
Rows with positive reconstructed car clients : 15,436
Car pooling size (persons/trip)             : 4.0
Trips per pooled group per year (driver)    : 19

VOT mode                                   : spatial_income
VOT USD/h min/median/mean/p95/max         : 0.021667 / 0.032500 / 0.029793 / 0.032500 / 0.054167
TODO: review MINIMUM_WAGE_USD_PER_MONTH and WAGE_RANGE 

In [ ]:
"""
Final QA stats for 4.6 raster output.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import rasterio

DATA_DIR = Path("dataset_big")
OUTPUT_RASTER = DATA_DIR / "real_end_user_price.tif"
SENTINEL = 9999.0
NODATA_FLOAT = -9999.0

print("[1/3] Loading output raster...")
if not OUTPUT_RASTER.exists():
    raise FileNotFoundError(f"Output file not found: {OUTPUT_RASTER}")

with rasterio.open(OUTPUT_RASTER) as src:
    arr = src.read().astype(np.float32)
    desc = list(src.descriptions)
    nodata = src.nodata

if nodata is None:
    nodata = NODATA_FLOAT

arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
name_to_idx = {d: i for i, d in enumerate(desc) if d is not None}

required = [
    "car_share",
    "walk_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_reseller_id_car",
    "best_time_car_min",
    "res_cost_kg_out_walk_ref",
    "res_cost_kg_out_car_ref",
    "cost_kg_walker",
    "cost_kg_driver",
]
missing = [k for k in required if k not in name_to_idx]
if missing:
    raise KeyError(f"Missing required bands: {missing}")

cost_walk = arr[name_to_idx["cost_kg_walker"]].astype(np.float64)
cost_car = arr[name_to_idx["cost_kg_driver"]].astype(np.float64)
ref_walk = arr[name_to_idx["res_cost_kg_out_walk_ref"]].astype(np.float64)
ref_car = arr[name_to_idx["res_cost_kg_out_car_ref"]].astype(np.float64)

print("[2/3] Computing QA metrics...")
walk_finite = np.isfinite(cost_walk)
car_finite = np.isfinite(cost_car)
walk_sentinel = walk_finite & np.isclose(cost_walk, SENTINEL)
car_sentinel = car_finite & np.isclose(cost_car, SENTINEL)
walk_valid = walk_finite & (~walk_sentinel)
car_valid = car_finite & (~car_sentinel)

walk_component = cost_walk - ref_walk
car_component = cost_car - ref_car
walk_component_valid = walk_valid & np.isfinite(ref_walk) & np.isfinite(walk_component)
car_component_valid = car_valid & np.isfinite(ref_car) & np.isfinite(car_component)

n = cost_walk.size
print("=== END USER PRICE RASTER QA ===")
print(f"Rows total (pixels)                     : {n:,}")
print(f"Walker finite cost                      : {int(walk_finite.sum()):,}/{n:,}")
print(f"Driver finite cost                      : {int(car_finite.sum()):,}/{n:,}")
print(f"Walker valid (cost != {SENTINEL:.0f})            : {int(walk_valid.sum()):,}/{n:,}")
print(f"Driver valid (cost != {SENTINEL:.0f})            : {int(car_valid.sum()):,}/{n:,}")
print(f"Walker sentinel ({SENTINEL:.0f}) count           : {int(walk_sentinel.sum()):,}/{n:,}")
print(f"Driver sentinel ({SENTINEL:.0f}) count           : {int(car_sentinel.sum()):,}/{n:,}")

if walk_valid.any():
    w = cost_walk[walk_valid]
    print("Walker cost min/median/mean/p95/max:",
          f"{np.nanmin(w):.4f} / {np.nanmedian(w):.4f} / {np.nanmean(w):.4f} / {np.nanpercentile(w, 95):.4f} / {np.nanmax(w):.4f}")
if car_valid.any():
    c = cost_car[car_valid]
    print("Driver cost min/median/mean/p95/max:",
          f"{np.nanmin(c):.4f} / {np.nanmedian(c):.4f} / {np.nanmean(c):.4f} / {np.nanpercentile(c, 95):.4f} / {np.nanmax(c):.4f}")

if walk_component_valid.any():
    wc = walk_component[walk_component_valid]
    print("Walker component min/median/mean/p95/max:",
          f"{np.nanmin(wc):.6f} / {np.nanmedian(wc):.6f} / {np.nanmean(wc):.6f} / {np.nanpercentile(wc, 95):.6f} / {np.nanmax(wc):.6f}")
if car_component_valid.any():
    dc = car_component[car_component_valid]
    print("Driver component min/median/mean/p95/max:",
          f"{np.nanmin(dc):.6f} / {np.nanmedian(dc):.6f} / {np.nanmean(dc):.6f} / {np.nanpercentile(dc, 95):.6f} / {np.nanmax(dc):.6f}")

print("[3/3] QA completed.")
print(f"Output checked: {OUTPUT_RASTER}")

[1/3] Loading output raster...
[2/3] Computing QA metrics...
=== END USER PRICE RASTER QA ===
Rows total (pixels)                     : 1,450,645
Walker finite cost                      : 1,450,645/1,450,645
Driver finite cost                      : 1,450,645/1,450,645
Walker valid (cost != 9999)            : 12,847/1,450,645
Driver valid (cost != 9999)            : 15,436/1,450,645
Walker sentinel (9999) count           : 1,437,798/1,450,645
Driver sentinel (9999) count           : 1,435,209/1,450,645
Walker cost min/median/mean/p95/max: 0.5441 / 0.6024 / 0.6181 / 0.6801 / 5.7408
Driver cost min/median/mean/p95/max: 0.5441 / 0.7053 / 0.8103 / 1.2938 / 1.9603
Walker component min/median/mean/p95/max: 0.000476 / 0.002299 / 0.003102 / 0.003811 / 5.157903
Driver component min/median/mean/p95/max: 0.000606 / 0.070972 / 0.193275 / 0.645436 / 0.805838
[3/3] QA completed.
Output checked: dataset_big\end_user_price.tif
